# Notebook 6 v2: Face-Cropped 4-Class Engagement Models

## What Failed in v1 (and Why)
The original NB6 achieved **0.3% engagement accuracy** (essentially random) because:
1. **Full-frame ImageNet features** — Swin-T/EfficientNet saw desks, walls, rooms — NOT facial expressions
2. **Extreme class imbalance** — Balanced class weights **over-corrected**, causing inverted majority-class collapse
3. **Frozen backbone** — ImageNet features carry zero engagement signal for faces

## v2 Fixes (This Notebook)
1. **MTCNN Face Detection + Cropping** — Detect face bounding box, crop face region with padding. Backbone now sees FACES, not furniture
2. **Focal Loss (γ=2)** — Handles extreme imbalance by down-weighting easy/well-classified examples, NOT by up-weighting rare classes
3. **WeightedRandomSampler** — Balanced mini-batches via oversampling (not loss weighting)
4. **E2E Fine-Tuning (enabled by default)** — Backbone learns face-expression features adapted to engagement
5. **Mixed Precision (AMP)** — 2× speedup for E2E, fits in T4 VRAM
6. **Strong Augmentation** — H-flip, color jitter, random erasing, temporal jitter, MixUp

## Expected Results
- v1 frozen features on full frames: **0.3%** engagement accuracy
- v2 face crops + focal loss + E2E: **target 40-55%** engagement accuracy
- Reference: CORAL with OpenFace AUs (NB1) = 48.3%, ViBED-Net SOTA = 73.4%

## Kaggle Setup
- Dataset: `seversnape/daisee-videos`
- GPU: **T4 x2** (required for E2E fine-tuning speed)
- Internet: **ON** (pretrained weights + facenet-pytorch)
- Estimated runtime: **4-6 hours**

In [ ]:
# ============================================================
# Cell 2: Installs & Imports
# ============================================================
import os
# Reduce noisy ffmpeg decode logs from broken DAiSEE clips
os.environ['OPENCV_FFMPEG_CAPTURE_OPTIONS'] = 'loglevel;error'

import subprocess, sys

# Install facenet-pytorch without pulling numpy>=2 (Kaggle has 1.26.4 + torch 2.2.2)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'scikit-learn', 'tqdm'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '--no-deps', 'facenet-pytorch'])

import glob, json, time, warnings, random, pickle, gc
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.amp import autocast, GradScaler

import torchvision.models as models
import torchvision.transforms as T

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, cohen_kappa_score
)
from sklearn.utils.class_weight import compute_class_weight

try:
    cv2.setLogLevel(cv2.LOG_LEVEL_ERROR)
except Exception:
    pass

try:
    from facenet_pytorch import MTCNN
    MTCNN_AVAILABLE = True
    print('MTCNN face detection loaded.')
except ImportError:
    MTCNN_AVAILABLE = False
    print('WARNING: facenet-pytorch not available. Using center-crop fallback.')

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}, GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================
class CFG:
    # --- Paths (auto-detected below) ---
    DAISEE_DIR = None
    VIDEO_DIR = None
    LABELS_DIR = None

    # --- Output ---
    MODEL_DIR = '/kaggle/working/trained_models'
    RESULTS_DIR = '/kaggle/working/experiment_results'
    FEATURE_DIR = '/kaggle/working/features_4class'
    FACE_CACHE = '/kaggle/working/face_bboxes.pkl'
    E2E_CKPT_DIR = '/kaggle/working/e2e_checkpoints'

    # --- Frame extraction ---
    N_FRAMES = 16
    IMG_SIZE = 224

    # --- Face Detection ---
    FACE_DETECT_FRAMES = 3
    FACE_PAD_RATIO = 0.3
    FACE_MIN_CONF = 0.80
    CENTER_CROP_FALLBACK = True

    # --- Frozen-feature training ---
    BATCH_SIZE = 64
    EPOCHS = 100
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 20
    NUM_CLASSES = 4
    DROPOUT = 0.4
    HIDDEN_DIM = 256
    LSTM_LAYERS = 2

    # --- Focal Loss ---
    FOCAL_GAMMA = 2.0

    # --- Oversampling ---
    OVERSAMPLE = True

    # --- MixUp ---
    MIXUP_ALPHA = 0.2

    # --- End-to-end Fine-Tuning (resume-safe) ---
    E2E_ENABLED = True
    E2E_EPOCHS = 20
    E2E_LR = 8e-5
    E2E_BATCH = 8
    E2E_GRAD_ACCUM = 4
    E2E_PATIENCE = 8
    E2E_MAX_HOURS = 5.0

    # Run only critical dimensions first to finish within Kaggle limits
    RUN_DIMS = ['engagement']
    # Set to [] to run all dims, e.g. RUN_DIMS = []

    # Resume controls
    E2E_RESUME = True
    E2E_SKIP_COMPLETED = True

    DIMS = ['Boredom', 'Engagement', 'Confusion', 'Frustration']
    DIM_NAMES = ['boredom', 'engagement', 'confusion', 'frustration']

os.makedirs(CFG.MODEL_DIR, exist_ok=True)
os.makedirs(CFG.RESULTS_DIR, exist_ok=True)
os.makedirs(CFG.FEATURE_DIR, exist_ok=True)
os.makedirs(CFG.E2E_CKPT_DIR, exist_ok=True)
print('Config ready.')

In [ ]:
# ============================================================
# Cell 4: Dataset Discovery & Label Loading
# ============================================================

def find_daisee_root():
    """Auto-detect DAiSEE dataset location on Kaggle."""
    candidates = glob.glob('/kaggle/input/**/AllLabels.csv', recursive=True)
    if not candidates:
        candidates = glob.glob('/kaggle/input/**/TrainLabels.csv', recursive=True)
    if not candidates:
        raise FileNotFoundError(
            'DAiSEE labels not found! Add seversnape/daisee-videos as input dataset.'
        )
    label_path = candidates[0]
    # Walk up to find the DAiSEE root
    parts = label_path.split(os.sep)
    for i, p in enumerate(parts):
        if p == 'DAiSEE':
            return os.sep.join(parts[:i+1])
    # Fallback: parent of Labels dir
    return str(Path(label_path).parent.parent)

def load_labels(csv_path):
    """Load DAiSEE labels CSV → {clip_id: [B, E, C, F]}."""
    df = pd.read_csv(csv_path)
    cols = {c.strip().lower(): c for c in df.columns}
    clip_col = cols.get('clipid', df.columns[0])
    labels = {}
    for _, row in df.iterrows():
        cid = str(row[clip_col]).replace('.avi', '').replace('.mp4', '').strip()
        labels[cid] = [
            int(row[cols.get('boredom', 'Boredom')]),
            int(row[cols.get('engagement', 'Engagement')]),
            int(row[cols.get('confusion', 'Confusion')]),
            int(row[cols.get('frustration', 'Frustration')]),
        ]
    return labels

def find_videos(video_dir):
    """Find all video files → {clip_id: path}."""
    videos = {}
    for ext in ('*.avi', '*.mp4'):
        for p in Path(video_dir).rglob(ext):
            videos[p.stem] = str(p)
    return videos

# --- Auto-detect paths ---
daisee_root = find_daisee_root()
CFG.DAISEE_DIR = daisee_root
CFG.VIDEO_DIR = os.path.join(daisee_root, 'DataSet')
CFG.LABELS_DIR = os.path.join(daisee_root, 'Labels')
print(f'DAiSEE root: {CFG.DAISEE_DIR}')
print(f'Video dir:   {CFG.VIDEO_DIR}')
print(f'Labels dir:  {CFG.LABELS_DIR}')

# --- Load labels for each split ---
train_labels = load_labels(os.path.join(CFG.LABELS_DIR, 'TrainLabels.csv'))
val_labels   = load_labels(os.path.join(CFG.LABELS_DIR, 'ValidationLabels.csv'))
test_labels  = load_labels(os.path.join(CFG.LABELS_DIR, 'TestLabels.csv'))

print(f'Labels: train={len(train_labels)}, val={len(val_labels)}, test={len(test_labels)}')

# --- Find video files ---
all_videos = {}
for split in ('Train', 'Validation', 'Test'):
    split_dir = os.path.join(CFG.VIDEO_DIR, split)
    if os.path.isdir(split_dir):
        vids = find_videos(split_dir)
        all_videos.update(vids)
        print(f'  {split}: {len(vids)} videos found')

# Match labels to videos
def match_split(labels_dict):
    matched = []
    for cid, lbl in labels_dict.items():
        if cid in all_videos:
            matched.append((cid, all_videos[cid], lbl))
    return matched

train_data = match_split(train_labels)
val_data   = match_split(val_labels)
test_data  = match_split(test_labels)

print(f'\nMatched: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}')

# Show class distribution for Engagement (4-class)
for split_name, data in [('Train', train_data), ('Val', val_data), ('Test', test_data)]:
    eng_labels = [d[2][1] for d in data]  # Engagement is index 1
    dist = {i: eng_labels.count(i) for i in range(4)}
    print(f'  {split_name} Engagement dist: {dist}')

In [ ]:
# ============================================================
# Cell 5: MTCNN Face Detection + Face-Aware Frame Extraction
# ============================================================
# Key insight: DAiSEE videos are webcam recordings of students.
# The face is the ONLY region carrying engagement signal.
# ImageNet backbones on full frames see desks/walls → useless.
# Cropping to face region is the single most important fix.

# --- Face detection ---
def detect_face_bbox(video_path, n_detect_frames=3, mtcnn_model=None):
    """
    Detect face in a few frames of the video.
    Returns (x1, y1, x2, y2) in normalized [0,1] coords, or None.
    Strategy: sample 3 frames from middle 60% of video, keep best detection.
    """
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None

    detect_indices = np.linspace(
        int(total * 0.2), int(total * 0.8), n_detect_frames, dtype=int
    )
    best_bbox = None
    best_conf = 0

    for idx in detect_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        h, w = frame.shape[:2]
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(frame_rgb)

        try:
            boxes, probs = mtcnn_model.detect(pil_img)
        except Exception:
            continue

        if boxes is not None and len(boxes) > 0 and probs[0] is not None:
            conf = float(probs[0])
            if conf > CFG.FACE_MIN_CONF and conf > best_conf:
                best_conf = conf
                bx = boxes[0]
                best_bbox = (
                    max(0, bx[0] / w), max(0, bx[1] / h),
                    min(1, bx[2] / w), min(1, bx[3] / h),
                )

    cap.release()
    return best_bbox


def pad_bbox(bbox, pad_ratio=0.3):
    """Add padding around bbox, clamp to [0,1]."""
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    px, py = bw * pad_ratio, bh * pad_ratio
    return (
        max(0, x1 - px), max(0, y1 - py),
        min(1, x2 + px), min(1, y2 + py),
    )


# --- Run face detection on all clips ---
face_bboxes = {}

if MTCNN_AVAILABLE:
    mtcnn = MTCNN(
        image_size=160, margin=0, min_face_size=40,
        thresholds=[0.6, 0.7, 0.7], factor=0.709,
        post_process=False, device=DEVICE, keep_all=False,
    )

    # Check if cached
    if os.path.exists(CFG.FACE_CACHE):
        with open(CFG.FACE_CACHE, 'rb') as f:
            face_bboxes = pickle.load(f)
        print(f'Loaded {len(face_bboxes)} cached face bboxes.')
    else:
        print('Detecting faces in all clips (one-time, ~20 min)...')
        all_clips = train_data + val_data + test_data
        found, missed = 0, 0

        for cid, vpath, lbl in tqdm(all_clips, desc='MTCNN face detection'):
            bbox = detect_face_bbox(vpath, CFG.FACE_DETECT_FRAMES, mtcnn)
            if bbox is not None:
                face_bboxes[cid] = pad_bbox(bbox, CFG.FACE_PAD_RATIO)
                found += 1
            else:
                missed += 1

        print(f'Faces found: {found}/{len(all_clips)} ({100*found/len(all_clips):.1f}%)')
        print(f'Faces missed: {missed} → center-crop fallback')

        with open(CFG.FACE_CACHE, 'wb') as f:
            pickle.dump(face_bboxes, f)
        print(f'Face bboxes cached to {CFG.FACE_CACHE}')

    del mtcnn
    torch.cuda.empty_cache()
else:
    print('No MTCNN — all clips will use center-crop fallback.')


# --- Face-aware frame extraction ---
def extract_frames(video_path, clip_id, n_frames=16, size=224):
    """
    Extract n_frames from video with FACE CROPPING.
    If face bbox exists: crop to face region.
    Otherwise: center-upper crop (DAiSEE faces are usually centered).
    """
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None

    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    bbox = face_bboxes.get(clip_id, None)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue

        h, w = frame.shape[:2]
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        if bbox is not None:
            # Crop to detected face region
            x1 = int(bbox[0] * w)
            y1 = int(bbox[1] * h)
            x2 = int(bbox[2] * w)
            y2 = int(bbox[3] * h)
            crop = frame[max(0,y1):max(1,y2), max(0,x1):max(1,x2)]
            if crop.size > 0 and crop.shape[0] > 5 and crop.shape[1] > 5:
                frame = crop
            else:
                # Fallback for bad crop
                ch, cw = int(h*0.7), int(w*0.7)
                ys, xs = max(0, h//6), max(0, (w-cw)//2)
                frame = frame[ys:ys+ch, xs:xs+cw]
        elif CFG.CENTER_CROP_FALLBACK:
            # Center-upper crop: top 70% height, center 70% width
            ch, cw = int(h * 0.7), int(w * 0.7)
            ys = max(0, h // 6)   # bias toward top (face region)
            xs = (w - cw) // 2
            frame = frame[ys:ys+ch, xs:xs+cw]

        frame = cv2.resize(frame, (size, size))
        frames.append(frame)

    cap.release()

    if len(frames) < n_frames // 2:
        return None
    while len(frames) < n_frames:
        frames.append(frames[-1])

    return np.stack(frames[:n_frames])

print(f'Face-aware frame extraction ready. '
      f'{len(face_bboxes)} clips have face bboxes.')

In [ ]:
# ============================================================
# Cell 6: Streaming Backbone Feature Extraction (Face Crops)
# ============================================================
# Same streaming approach as v1 (constant RAM), but now extracts
# FACE CROPS instead of full frames.

def get_backbone(name='swin_t'):
    """Load pretrained backbone, remove classification head."""
    if name == 'swin_t':
        weights = models.Swin_T_Weights.IMAGENET1K_V1
        model = models.swin_t(weights=weights)
        model.head = nn.Identity()
        feature_dim = 768
        transform = weights.transforms()
    elif name == 'efficientnet_v2_s':
        weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        model = models.efficientnet_v2_s(weights=weights)
        model.classifier = nn.Identity()
        feature_dim = 1280
        transform = weights.transforms()
    else:
        raise ValueError(f'Unknown backbone: {name}')

    model = model.to(DEVICE).eval()
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model, feature_dim, transform


@torch.no_grad()
def stream_extract_features(data_list, backbone, transform, n_frames, img_size):
    """
    Stream videos one-by-one with FACE CROPS: read frames → crop face →
    backbone → features → discard. Constant RAM usage.
    """
    clip_ids_out, features_list, labels_list = [], [], []
    failed = 0

    for cid, vpath, lbl in tqdm(data_list, desc='Extract (face-crop)'):
        frames = extract_frames(vpath, cid, n_frames, img_size)
        if frames is None:
            failed += 1
            continue

        pil_frames = [Image.fromarray(f) for f in frames]
        batch = torch.stack([transform(img) for img in pil_frames]).to(DEVICE)

        feats = backbone(batch)  # (n_frames, feat_dim)
        features_list.append(feats.cpu().numpy())
        clip_ids_out.append(cid)
        labels_list.append(lbl)

    if failed > 0:
        print(f'  Failed: {failed}/{len(data_list)} clips')

    features = np.stack(features_list)
    labels = np.array(labels_list)
    return clip_ids_out, features, labels


def extract_and_save(backbone_name, train_data, val_data, test_data):
    """Extract face-crop features for all splits."""
    # Check for cached features
    cache_path = os.path.join(CFG.FEATURE_DIR, f'{backbone_name}_train_face.npy')
    if os.path.exists(cache_path):
        print(f'\nLoading cached {backbone_name} face-crop features...')
        X_train = np.load(os.path.join(CFG.FEATURE_DIR, f'{backbone_name}_train_face.npy'))
        X_val = np.load(os.path.join(CFG.FEATURE_DIR, f'{backbone_name}_val_face.npy'))
        X_test = np.load(os.path.join(CFG.FEATURE_DIR, f'{backbone_name}_test_face.npy'))
        train_ids = [d[0] for d in train_data]
        val_ids = [d[0] for d in val_data]
        test_ids = [d[0] for d in test_data]
        train_lbls = np.array([d[2] for d in train_data])
        val_lbls = np.array([d[2] for d in val_data])
        test_lbls = np.array([d[2] for d in test_data])
        feat_dim = X_train.shape[-1]
        print(f'  Loaded: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')
        return (train_ids, val_ids, test_ids,
                X_train, X_val, X_test,
                train_lbls, val_lbls, test_lbls, feat_dim)

    print(f'\n{"="*60}')
    print(f'Extracting FACE-CROP features with {backbone_name}...')
    print(f'{"="*60}')

    backbone, feat_dim, transform = get_backbone(backbone_name)

    t0 = time.time()
    train_ids, X_train, train_lbls = stream_extract_features(
        train_data, backbone, transform, CFG.N_FRAMES, CFG.IMG_SIZE)
    val_ids, X_val, val_lbls = stream_extract_features(
        val_data, backbone, transform, CFG.N_FRAMES, CFG.IMG_SIZE)
    test_ids, X_test, test_lbls = stream_extract_features(
        test_data, backbone, transform, CFG.N_FRAMES, CFG.IMG_SIZE)

    print(f'Extraction done in {time.time()-t0:.0f}s')
    print(f'Feature shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')

    # Cache to disk
    np.save(os.path.join(CFG.FEATURE_DIR, f'{backbone_name}_train_face.npy'), X_train)
    np.save(os.path.join(CFG.FEATURE_DIR, f'{backbone_name}_val_face.npy'), X_val)
    np.save(os.path.join(CFG.FEATURE_DIR, f'{backbone_name}_test_face.npy'), X_test)

    del backbone
    torch.cuda.empty_cache(); gc.collect()

    return (train_ids, val_ids, test_ids,
            X_train, X_val, X_test,
            train_lbls, val_lbls, test_lbls, feat_dim)


# --- Extract with both backbones (face-cropped) ---
(train_ids, val_ids, test_ids,
 swin_train, swin_val, swin_test,
 train_lbls, val_lbls, test_lbls,
 swin_dim) = extract_and_save('swin_t', train_data, val_data, test_data)

(_, _, _,
 effnet_train, effnet_val, effnet_test,
 _, _, _,
 effnet_dim) = extract_and_save('efficientnet_v2_s', train_data, val_data, test_data)

print(f'\nSwin-T dim: {swin_dim}, EfficientNetV2-S dim: {effnet_dim}')
print(f'Clips: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}')

In [ ]:
# ============================================================
# Cell 7: Model Definition + Focal Loss
# ============================================================

class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017) for extreme class imbalance.
    Down-weights well-classified examples, focuses on hard examples.
    Much better than balanced class weights which over-correct.
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # per-class weights (optional, mild)
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)  # probability of correct class
        focal = ((1 - pt) ** self.gamma) * ce

        if self.reduction == 'mean':
            return focal.mean()
        elif self.reduction == 'sum':
            return focal.sum()
        return focal


class TemporalAttentionClassifier(nn.Module):
    """
    BiLSTM + self-attention over frame features → 4-class classifier.
    Improvements over v1: more dropout, residual connection, layer norm.
    """
    def __init__(self, feature_dim, hidden_dim=256, num_layers=2,
                 num_classes=4, dropout=0.4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(
            hidden_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        attn_w = torch.softmax(self.attention(lstm_out), dim=1)
        context = (lstm_out * attn_w).sum(dim=1)
        return self.classifier(context)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# Quick check
_m = TemporalAttentionClassifier(768, 256, 2, 4, 0.4).to(DEVICE)
_x = torch.randn(4, CFG.N_FRAMES, 768).to(DEVICE)
print(f'Temporal model: {count_params(_m):,} params, '
      f'Input: {_x.shape} → Output: {_m(_x).shape}')
del _m, _x

In [ ]:
# ============================================================
# Cell 8: Training Utilities (Oversampling, MixUp, Focal Loss)
# ============================================================

class FeatureDataset(Dataset):
    """Pre-extracted clip features + 4-class label for one dimension."""
    def __init__(self, features, labels, dim_idx, augment=False):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels[:, dim_idx])
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.features[idx].clone()
        if self.augment:
            # Temporal jitter: drop 1-2 frames, pad with repeats
            if random.random() < 0.3:
                T = x.shape[0]
                drop = random.randint(1, min(2, T // 4))
                keep = sorted(random.sample(range(T), T - drop))
                x_k = x[keep]
                x = torch.cat([x_k, x_k[-drop:]], dim=0)[:T]
            # Feature noise
            if random.random() < 0.3:
                x = x + torch.randn_like(x) * 0.02
            # Temporal reversal
            if random.random() < 0.2:
                x = x.flip(0)
        return x, self.labels[idx]


def make_sampler(labels, dim_idx):
    """WeightedRandomSampler for balanced mini-batches."""
    y = labels[:, dim_idx]
    counts = Counter(y)
    total = len(y)
    class_weights = {c: total / (len(counts) * n) for c, n in counts.items()}
    sample_weights = torch.DoubleTensor([class_weights[int(yi)] for yi in y])
    return WeightedRandomSampler(sample_weights, num_samples=len(y), replacement=True)


def get_mild_class_weights(labels, dim_idx, num_classes=4, smoothing=0.5):
    """
    Mild class weights (sqrt of balanced) to avoid over-correction.
    v1 used full balanced weights which caused inverted collapse.
    """
    y = labels[:, dim_idx]
    counts = Counter(y)
    total = len(y)
    weights = torch.ones(num_classes).to(DEVICE)
    for c in range(num_classes):
        if c in counts and counts[c] > 0:
            w = total / (num_classes * counts[c])
            weights[c] = w ** smoothing  # sqrt dampening
    return weights


def mixup_data(x, y, alpha=0.2):
    """MixUp: interpolate between pairs of samples."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)  # ensure lam >= 0.5
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, criterion, optimizer, scheduler=None,
                    use_mixup=False, mixup_alpha=0.2):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for features, labels in loader:
        features, labels = features.to(DEVICE), labels.to(DEVICE)

        if use_mixup and random.random() < 0.5:
            features, y_a, y_b, lam = mixup_data(features, labels, mixup_alpha)
            logits = model(features)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
            correct += (lam * (logits.argmax(1) == y_a).float()
                        + (1-lam) * (logits.argmax(1) == y_b).float()).sum().item()
        else:
            logits = model(features)
            loss = criterion(logits, labels)
            correct += (logits.argmax(1) == labels).sum().item()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)

    if scheduler:
        scheduler.step()
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for features, labels in loader:
        features = features.to(DEVICE)
        logits = model(features)
        probs = torch.softmax(logits, dim=1)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(labels)
        all_probs.append(probs.cpu())
    return (
        torch.cat(all_preds).numpy(),
        torch.cat(all_labels).numpy(),
        torch.cat(all_probs).numpy(),
    )


def evaluate_metrics(y_true, y_pred, y_prob, dim_name):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')

    y_true_bin = (y_true >= 2).astype(int)
    y_pred_bin = (y_pred >= 2).astype(int)
    p_high = y_prob[:, 2:].sum(axis=1)
    f1_bin = f1_score(y_true_bin, y_pred_bin, average='macro', zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    print(f'\n{dim_name.upper()} Results:')
    print(f'  4c Accuracy:  {acc:.4f}')
    print(f'  4c F1-macro:  {f1m:.4f}')
    print(f'  QWK:          {qwk:.4f}')
    print(f'  Binary F1m:   {f1_bin:.4f}')
    print(f'  Confusion matrix:\n{cm}')

    return {
        'accuracy_4class': float(acc),
        'f1_macro_4class': float(f1m),
        'f1_weighted_4class': float(f1w),
        'qwk': float(qwk),
        'f1_binary_equiv': float(f1_bin),
    }, p_high


print('Training utilities ready (Focal Loss + Oversampling + MixUp).')

In [ ]:
# ============================================================
# Cell 9: Train Frozen-Feature Models (Focal Loss + Oversampling)
# ============================================================

def train_temporal_model(backbone_name, X_train, X_val, X_test,
                         train_lbls, val_lbls, test_lbls, feature_dim):
    """Train 4-class temporal model for all dimensions with v2 fixes."""
    all_results, all_probs, all_binary = {}, {}, {}

    for dim_idx, dim_name in enumerate(CFG.DIM_NAMES):
        print(f'\n{"="*60}')
        print(f'{backbone_name} → {dim_name} (4-class, face-crop + focal)')
        print(f'{"="*60}')

        # Show class distribution
        y_train = train_lbls[:, dim_idx]
        dist = Counter(y_train)
        print(f'  Train dist: {dict(sorted(dist.items()))}')

        # Datasets
        ds_train = FeatureDataset(X_train, train_lbls, dim_idx, augment=True)
        ds_val   = FeatureDataset(X_val, val_lbls, dim_idx, augment=False)
        ds_test  = FeatureDataset(X_test, test_lbls, dim_idx, augment=False)

        # Oversampling → balanced mini-batches
        if CFG.OVERSAMPLE:
            sampler = make_sampler(train_lbls, dim_idx)
            dl_train = DataLoader(ds_train, batch_size=CFG.BATCH_SIZE,
                                  sampler=sampler, num_workers=2,
                                  pin_memory=True, drop_last=True)
        else:
            dl_train = DataLoader(ds_train, batch_size=CFG.BATCH_SIZE,
                                  shuffle=True, num_workers=2, pin_memory=True)

        dl_val  = DataLoader(ds_val, batch_size=256, num_workers=2, pin_memory=True)
        dl_test = DataLoader(ds_test, batch_size=256, num_workers=2, pin_memory=True)

        # Model
        model = TemporalAttentionClassifier(
            feature_dim, CFG.HIDDEN_DIM, CFG.LSTM_LAYERS,
            CFG.NUM_CLASSES, CFG.DROPOUT,
        ).to(DEVICE)

        # Focal Loss (mild class weights to help, but NOT full balanced)
        alpha = get_mild_class_weights(train_lbls, dim_idx, smoothing=0.3)
        criterion = FocalLoss(gamma=CFG.FOCAL_GAMMA, alpha=alpha)

        optimizer = torch.optim.AdamW(
            model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

        best_val_f1, best_state, patience_ctr = 0, None, 0

        for epoch in range(CFG.EPOCHS):
            train_loss, train_acc = train_one_epoch(
                model, dl_train, criterion, optimizer, scheduler,
                use_mixup=True, mixup_alpha=CFG.MIXUP_ALPHA)

            val_preds, val_true, val_probs = evaluate(model, dl_val)
            val_acc = accuracy_score(val_true, val_preds)
            val_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f'  Epoch {epoch+1}/{CFG.EPOCHS}: '
                      f'loss={train_loss:.4f} t_acc={train_acc:.3f} '
                      f'v_acc={val_acc:.3f} v_f1m={val_f1:.3f}')

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1

            if patience_ctr >= CFG.PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

        # Evaluate on test
        model.load_state_dict(best_state)
        test_preds, test_true, test_probs = evaluate(model, dl_test)
        metrics, p_high = evaluate_metrics(test_true, test_preds, test_probs, dim_name)
        metrics['best_val_f1'] = float(best_val_f1)

        prefix = backbone_name.replace('_', '')
        torch.save(best_state, os.path.join(
            CFG.MODEL_DIR, f'{prefix}_temporal_{dim_name}.pt'))
        np.save(os.path.join(
            CFG.MODEL_DIR, f'proba_{prefix}4c_{dim_name}.npy'), test_probs)
        np.save(os.path.join(
            CFG.MODEL_DIR, f'labels_4class_{dim_name}.npy'), test_true)
        np.save(os.path.join(
            CFG.MODEL_DIR, f'proba_{prefix}_{dim_name}.npy'), p_high)

        all_results[dim_name] = metrics
        all_probs[dim_name] = test_probs
        all_binary[dim_name] = p_high

        del model, best_state; torch.cuda.empty_cache()

    return all_results, all_probs, all_binary


# === TRAIN SWIN-T FROZEN MODELS ===
print('\n' + '='*60)
print('TRAINING SWIN-T + LSTM (FACE-CROP + FOCAL LOSS)')
print('='*60)
swin_results, swin_probs_4c, swin_probs_bin = train_temporal_model(
    'swin_t', swin_train, swin_val, swin_test,
    train_lbls, val_lbls, test_lbls, swin_dim)

print('\n' + '='*60)
print('SWIN-T RESULTS SUMMARY (v2 — face crops)')
print('='*60)
for dim, m in swin_results.items():
    print(f"  {dim:>12s}: 4c_acc={m['accuracy_4class']:.4f}  "
          f"f1m={m['f1_macro_4class']:.4f}  "
          f"bin_f1m={m['f1_binary_equiv']:.4f}  qwk={m['qwk']:.4f}")

In [ ]:
# ============================================================
# Cell 10: Train EfficientNetV2-S Frozen Models
# ============================================================

print('\n' + '='*60)
print('TRAINING EFFICIENTNETV2-S + LSTM (FACE-CROP + FOCAL LOSS)')
print('='*60)
effnet_results, effnet_probs_4c, effnet_probs_bin = train_temporal_model(
    'efficientnet_v2_s', effnet_train, effnet_val, effnet_test,
    train_lbls, val_lbls, test_lbls, effnet_dim)

print('\n' + '='*60)
print('EFFICIENTNETV2-S RESULTS SUMMARY (v2 — face crops)')
print('='*60)
for dim, m in effnet_results.items():
    print(f"  {dim:>12s}: 4c_acc={m['accuracy_4class']:.4f}  "
          f"f1m={m['f1_macro_4class']:.4f}  "
          f"bin_f1m={m['f1_binary_equiv']:.4f}  qwk={m['qwk']:.4f}")

In [ ]:
# ============================================================
# Cell 11: End-to-End Fine-Tuning (resume-safe + time-safe)
# ============================================================

class EndToEndModel(nn.Module):
    def __init__(self, backbone_name='swin_t', hidden_dim=256,
                 num_classes=4, dropout=0.4, n_frames=16):
        super().__init__()
        self.n_frames = n_frames

        if backbone_name == 'swin_t':
            weights = models.Swin_T_Weights.IMAGENET1K_V1
            self.backbone = models.swin_t(weights=weights)
            self.backbone.head = nn.Identity()
            feature_dim = 768
        elif backbone_name == 'efficientnet_v2_s':
            weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
            self.backbone = models.efficientnet_v2_s(weights=weights)
            self.backbone.classifier = nn.Identity()
            feature_dim = 1280
        else:
            raise ValueError(backbone_name)

        self.temporal = TemporalAttentionClassifier(
            feature_dim, hidden_dim, 2, num_classes, dropout)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        features = self.backbone(x)
        features = features.view(B, T, -1)
        return self.temporal(features)


class FaceCropVideoDataset(Dataset):
    def __init__(self, data_list, dim_idx, transform, n_frames=16, augment=False):
        self.data = data_list
        self.dim_idx = dim_idx
        self.transform = transform
        self.n_frames = n_frames
        self.augment = augment
        self.aug_transform = T.Compose([
            T.RandomHorizontalFlip(0.5),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
            T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
            T.RandomGrayscale(p=0.1),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        cid, vpath, lbl = self.data[idx]
        frames_np = extract_frames(vpath, cid, self.n_frames, CFG.IMG_SIZE)
        if frames_np is None:
            frames_np = np.zeros((self.n_frames, CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)

        tensors = []
        for f in frames_np:
            img = Image.fromarray(f)
            if self.augment:
                img = self.aug_transform(img)
            tensors.append(self.transform(img))

        return torch.stack(tensors), lbl[self.dim_idx]


def _save_e2e_ckpt(path, model, optimizer, scheduler, scaler,
                   epoch, best_val_f1, patience_ctr):
    raw = model.module if hasattr(model, 'module') else model
    torch.save({
        'model': raw.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'epoch': epoch,
        'best_val_f1': best_val_f1,
        'patience_ctr': patience_ctr,
    }, path)


def _load_e2e_ckpt(path, model, optimizer, scheduler, scaler):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    raw = model.module if hasattr(model, 'module') else model
    raw.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    return (
        int(ckpt.get('epoch', 0)) + 1,
        float(ckpt.get('best_val_f1', 0.0)),
        int(ckpt.get('patience_ctr', 0)),
    )


e2e_results = {}
e2e_probs_4c = {}
e2e_probs_bin = {}

if CFG.E2E_ENABLED:
    print('\n' + '='*60)
    print('END-TO-END FINE-TUNING (RESUME-SAFE)')
    print('='*60)

    e2e_backbone = 'swin_t'
    weights = models.Swin_T_Weights.IMAGENET1K_V1
    e2e_transform = weights.transforms()
    run_start = time.time()

    dim_iter = CFG.DIM_NAMES if not CFG.RUN_DIMS else CFG.RUN_DIMS

    for dim_name in dim_iter:
        dim_idx = CFG.DIM_NAMES.index(dim_name)

        final_prob_path = os.path.join(CFG.MODEL_DIR, f'proba_e2eswin4c_{dim_name}.npy')
        final_bin_path = os.path.join(CFG.MODEL_DIR, f'proba_e2eswin_{dim_name}.npy')
        final_model_path = os.path.join(CFG.MODEL_DIR, f'e2e_swin_{dim_name}.pt')
        ckpt_path = os.path.join(CFG.E2E_CKPT_DIR, f'e2e_{dim_name}_last.pt')

        if CFG.E2E_SKIP_COMPLETED and os.path.exists(final_prob_path) and os.path.exists(final_model_path):
            print(f'\n[SKIP] {dim_name}: final files already exist.')
            probs = np.load(final_prob_path)
            y_true = test_lbls[:len(probs), dim_idx]
            preds = probs.argmax(axis=1)
            metrics, p_high = evaluate_metrics(y_true, preds, probs, dim_name)
            e2e_results[dim_name] = metrics
            e2e_probs_4c[dim_name] = probs
            e2e_probs_bin[dim_name] = p_high
            continue

        print(f'\n{"="*60}')
        print(f'E2E {e2e_backbone} -> {dim_name} (face-crop)')
        print(f'{"="*60}')

        ds_train = FaceCropVideoDataset(train_data, dim_idx, e2e_transform, CFG.N_FRAMES, augment=True)
        ds_val = FaceCropVideoDataset(val_data, dim_idx, e2e_transform, CFG.N_FRAMES, augment=False)
        ds_test = FaceCropVideoDataset(test_data, dim_idx, e2e_transform, CFG.N_FRAMES, augment=False)

        if CFG.OVERSAMPLE:
            sampler = make_sampler(train_lbls, dim_idx)
            dl_train = DataLoader(ds_train, batch_size=CFG.E2E_BATCH,
                                  sampler=sampler, num_workers=2,
                                  pin_memory=True, drop_last=True)
        else:
            dl_train = DataLoader(ds_train, batch_size=CFG.E2E_BATCH,
                                  shuffle=True, num_workers=2, pin_memory=True)

        dl_val = DataLoader(ds_val, batch_size=CFG.E2E_BATCH, num_workers=2, pin_memory=True)
        dl_test = DataLoader(ds_test, batch_size=CFG.E2E_BATCH, num_workers=2, pin_memory=True)

        model = EndToEndModel(e2e_backbone, CFG.HIDDEN_DIM, 4, CFG.DROPOUT, CFG.N_FRAMES).to(DEVICE)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        print(f'  E2E params: {count_params(model):,}')

        raw = model.module if hasattr(model, 'module') else model
        optimizer = torch.optim.AdamW([
            {'params': raw.backbone.parameters(), 'lr': CFG.E2E_LR * 0.1},
            {'params': raw.temporal.parameters(), 'lr': CFG.E2E_LR},
        ], weight_decay=CFG.WEIGHT_DECAY)

        alpha = get_mild_class_weights(train_lbls, dim_idx, smoothing=0.3)
        criterion = FocalLoss(gamma=CFG.FOCAL_GAMMA, alpha=alpha)

        total_steps = max(1, len(dl_train) * CFG.E2E_EPOCHS // CFG.E2E_GRAD_ACCUM)
        scheduler = OneCycleLR(
            optimizer, max_lr=[CFG.E2E_LR * 0.1, CFG.E2E_LR],
            total_steps=total_steps, pct_start=0.1,
            anneal_strategy='cos', div_factor=10, final_div_factor=100)

        scaler = GradScaler('cuda')

        start_epoch = 0
        best_val_f1 = 0.0
        best_state = None
        patience_ctr = 0

        if CFG.E2E_RESUME and os.path.exists(ckpt_path):
            print(f'  Resuming from checkpoint: {ckpt_path}')
            start_epoch, best_val_f1, patience_ctr = _load_e2e_ckpt(
                ckpt_path, model, optimizer, scheduler, scaler)
            raw = model.module if hasattr(model, 'module') else model
            best_state = {k: v.clone() for k, v in raw.state_dict().items()}
            print(f'  Resume epoch={start_epoch}, best_val_f1={best_val_f1:.4f}, patience={patience_ctr}')

        for epoch in range(start_epoch, CFG.E2E_EPOCHS):
            if (time.time() - run_start) > CFG.E2E_MAX_HOURS * 3600:
                print('  Time budget reached. Saving checkpoint and stopping cleanly.')
                _save_e2e_ckpt(ckpt_path, model, optimizer, scheduler, scaler,
                               epoch, best_val_f1, patience_ctr)
                raise SystemExit('Stop now and rerun Cell 11 to resume automatically.')

            model.train()
            total_loss, total, correct = 0, 0, 0
            optimizer.zero_grad()

            for step, (frames, labels) in enumerate(tqdm(dl_train, desc=f'E{epoch+1}', leave=False)):
                frames, labels = frames.to(DEVICE), labels.to(DEVICE)

                with autocast('cuda'):
                    logits = model(frames)
                    loss = criterion(logits, labels) / CFG.E2E_GRAD_ACCUM

                scaler.scale(loss).backward()

                if (step + 1) % CFG.E2E_GRAD_ACCUM == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                total_loss += loss.item() * CFG.E2E_GRAD_ACCUM * labels.size(0)
                correct += (logits.argmax(1) == labels).sum().item()
                total += labels.size(0)

            val_preds, val_true, val_probs = evaluate(model, dl_val)
            val_acc = accuracy_score(val_true, val_preds)
            val_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)

            print(f'  Epoch {epoch+1}/{CFG.E2E_EPOCHS}: '
                  f'loss={total_loss/max(total,1):.4f} '
                  f't_acc={correct/max(total,1):.3f} '
                  f'v_acc={val_acc:.3f} v_f1m={val_f1:.3f}')

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                raw = model.module if hasattr(model, 'module') else model
                best_state = {k: v.clone() for k, v in raw.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1

            _save_e2e_ckpt(ckpt_path, model, optimizer, scheduler, scaler,
                           epoch, best_val_f1, patience_ctr)

            if patience_ctr >= CFG.E2E_PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

        raw = model.module if hasattr(model, 'module') else model
        if best_state is not None:
            raw.load_state_dict(best_state)

        test_preds, test_true, test_probs = evaluate(model, dl_test)
        metrics, p_high = evaluate_metrics(test_true, test_preds, test_probs, dim_name)

        torch.save(raw.state_dict(), final_model_path)
        np.save(final_prob_path, test_probs)
        np.save(final_bin_path, p_high)

        if os.path.exists(ckpt_path):
            os.remove(ckpt_path)

        e2e_results[dim_name] = metrics
        e2e_probs_4c[dim_name] = test_probs
        e2e_probs_bin[dim_name] = p_high

        del model, best_state
        torch.cuda.empty_cache()
        gc.collect()

    print('\n' + '='*60)
    print('E2E RESULTS (completed dimensions)')
    print('='*60)
    for dim, m in e2e_results.items():
        print(f"  {dim:>12s}: 4c_acc={m['accuracy_4class']:.4f}  "
              f"f1m={m['f1_macro_4class']:.4f}  "
              f"bin_f1m={m['f1_binary_equiv']:.4f}  qwk={m['qwk']:.4f}")
else:
    print('E2E fine-tuning DISABLED. Set CFG.E2E_ENABLED = True.')

In [ ]:
# ============================================================
# Cell 12: Test-Time Augmentation (TTA)
# ============================================================

def predict_with_tta(model, features, n_tta=5):
    """TTA: average predictions from augmented copies."""
    model.eval()
    all_probs = []

    with torch.no_grad():
        x = torch.FloatTensor(features).to(DEVICE)

        # 1. Original
        all_probs.append(torch.softmax(model(x), dim=1).cpu().numpy())

        # 2. Reversed temporal order
        all_probs.append(torch.softmax(model(x.flip(1)), dim=1).cpu().numpy())

        # 3-5. Random temporal shifts + noise
        for _ in range(n_tta - 2):
            shift = random.randint(1, max(1, x.shape[1] // 4))
            x_aug = torch.roll(x, shifts=shift, dims=1)
            x_aug = x_aug + torch.randn_like(x_aug) * 0.01
            all_probs.append(torch.softmax(model(x_aug), dim=1).cpu().numpy())

    return np.mean(all_probs, axis=0)


print('Applying TTA to frozen-feature models...')
tta_results = {}

for backbone_name, X_test_feat, feat_dim, prefix in [
    ('swin_t', swin_test, swin_dim, 'swint'),
    ('efficientnet_v2_s', effnet_test, effnet_dim, 'efficientnetv2s'),
]:
    print(f'\nTTA for {backbone_name}:')
    for dim_idx, dim_name in enumerate(CFG.DIM_NAMES):
        model = TemporalAttentionClassifier(
            feat_dim, CFG.HIDDEN_DIM, CFG.LSTM_LAYERS,
            CFG.NUM_CLASSES, CFG.DROPOUT).to(DEVICE)
        state = torch.load(
            os.path.join(CFG.MODEL_DIR, f'{prefix}_temporal_{dim_name}.pt'),
            map_location=DEVICE, weights_only=True)
        model.load_state_dict(state)

        tta_probs = predict_with_tta(model, X_test_feat, n_tta=5)
        tta_preds = tta_probs.argmax(axis=1)
        y_true = test_lbls[:len(tta_preds), dim_idx]

        acc = accuracy_score(y_true, tta_preds)
        f1m = f1_score(y_true, tta_preds, average='macro', zero_division=0)

        base = swin_results if backbone_name == 'swin_t' else effnet_results
        base_acc = base[dim_name]['accuracy_4class']
        print(f'  {dim_name}: acc={acc:.4f} (Δ{acc-base_acc:+.4f}) f1m={f1m:.4f}')

        np.save(os.path.join(
            CFG.MODEL_DIR, f'proba_{prefix}4c_tta_{dim_name}.npy'), tta_probs)
        np.save(os.path.join(
            CFG.MODEL_DIR, f'proba_{prefix}_tta_{dim_name}.npy'),
            tta_probs[:, 2:].sum(axis=1))

        tta_results[f'{backbone_name}_{dim_name}'] = {
            'accuracy_4class': float(acc), 'f1_macro_4class': float(f1m)}

        del model; torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Cell 13: Mega-Ensemble (All Streams: Frozen + E2E)
# ============================================================

print('='*60)
print('MEGA-ENSEMBLE: All available streams')
print('='*60)

ensemble_results = {}
for dim_idx, dim_name in enumerate(CFG.DIM_NAMES):
    # Collect all available probability streams
    streams = []
    stream_names = []

    # Frozen Swin-T
    if dim_name in swin_probs_4c:
        streams.append(swin_probs_4c[dim_name])
        stream_names.append('swin_frozen')

    # Frozen EfficientNet
    if dim_name in effnet_probs_4c:
        streams.append(effnet_probs_4c[dim_name])
        stream_names.append('effnet_frozen')

    # E2E fine-tuned
    if dim_name in e2e_probs_4c:
        # E2E gets 2x weight (more informative)
        streams.append(e2e_probs_4c[dim_name])
        streams.append(e2e_probs_4c[dim_name])
        stream_names.append('e2e (2x)')

    if len(streams) == 0:
        print(f'  {dim_name}: no streams available, skipping')
        continue

    min_len = min(len(s) for s in streams)
    ensemble_probs = np.mean([s[:min_len] for s in streams], axis=0)
    ensemble_preds = ensemble_probs.argmax(axis=1)
    y_true = test_lbls[:min_len, dim_idx]

    acc = accuracy_score(y_true, ensemble_preds)
    f1m = f1_score(y_true, ensemble_preds, average='macro', zero_division=0)
    qwk = cohen_kappa_score(y_true, ensemble_preds, weights='quadratic')

    p_high = ensemble_probs[:, 2:].sum(axis=1)
    y_true_bin = (y_true >= 2).astype(int)
    y_pred_bin = (ensemble_preds >= 2).astype(int)
    f1_bin = f1_score(y_true_bin, y_pred_bin, average='macro', zero_division=0)

    ensemble_results[dim_name] = {
        'accuracy_4class': float(acc),
        'f1_macro_4class': float(f1m),
        'qwk': float(qwk),
        'f1_binary_equiv': float(f1_bin),
        'streams': stream_names,
    }

    np.save(os.path.join(CFG.MODEL_DIR, f'proba_ensemble4c_{dim_name}.npy'),
            ensemble_probs)
    np.save(os.path.join(CFG.MODEL_DIR, f'proba_ensemble_{dim_name}.npy'),
            p_high)

    print(f'{dim_name:>12s}: 4c_acc={acc:.4f}  f1m={f1m:.4f}  '
          f'qwk={qwk:.4f}  bin_f1m={f1_bin:.4f}  '
          f'[{", ".join(stream_names)}]')

avg_acc = np.mean([m['accuracy_4class'] for m in ensemble_results.values()])
avg_f1 = np.mean([m['f1_macro_4class'] for m in ensemble_results.values()])
print(f'\n{"Avg":>12s}: 4c_acc={avg_acc:.4f}  f1m={avg_f1:.4f}')
print(f'\nPrevious v1 (full-frame, no face crop): Engagement acc = 0.3%')
print(f'CORAL (NB1, OpenFace AUs): Engagement acc = 48.3%')
eng_acc = ensemble_results.get("engagement", {}).get("accuracy_4class", 0)
print(f'This v2 ensemble: Engagement acc = {eng_acc*100:.1f}%')

In [ ]:
# ============================================================
# Cell 14: Save All Results + Comparison Table
# ============================================================

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results = {
    'experiment': 'v6_frame_4class_v2',
    'timestamp': timestamp,
    'changes_from_v1': [
        'MTCNN face detection + face cropping',
        'Focal loss (gamma=2) instead of weighted CE',
        'WeightedRandomSampler oversampling',
        'MixUp augmentation (alpha=0.2)',
        'E2E fine-tuning enabled with AMP',
        'Stronger data augmentation',
    ],
    'config': {
        'n_frames': CFG.N_FRAMES,
        'img_size': CFG.IMG_SIZE,
        'epochs': CFG.EPOCHS,
        'lr': CFG.LR,
        'focal_gamma': CFG.FOCAL_GAMMA,
        'mixup_alpha': CFG.MIXUP_ALPHA,
        'oversample': CFG.OVERSAMPLE,
        'e2e_enabled': CFG.E2E_ENABLED,
        'e2e_epochs': CFG.E2E_EPOCHS,
        'e2e_lr': CFG.E2E_LR,
        'face_pad_ratio': CFG.FACE_PAD_RATIO,
    },
    'data': {
        'train_clips': len(train_ids),
        'val_clips': len(val_ids),
        'test_clips': len(test_ids),
        'faces_detected': len(face_bboxes),
    },
    'swin_t_frozen': swin_results,
    'efficientnet_v2_s_frozen': effnet_results,
    'ensemble': ensemble_results,
    'tta_results': tta_results,
}
if e2e_results:
    results['e2e_swin_t'] = e2e_results

results_path = os.path.join(
    CFG.RESULTS_DIR, f'experiment_v6_4class_v2_{timestamp}.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {results_path}')

# List saved files
print('\nSaved model files:')
for fn in sorted(os.listdir(CFG.MODEL_DIR)):
    sz = os.path.getsize(os.path.join(CFG.MODEL_DIR, fn)) / 1024
    print(f'  {fn} ({sz:.0f} KB)')

# Comparison table
print('\n' + '='*70)
print('COMPARISON TABLE: v1 vs v2 vs SOTA')
print('='*70)
print(f'{"Method":>38s} | {"Eng 4c Acc":>10s} | {"Avg 4c Acc":>10s}')
print('-'*65)
print(f'{"v1 Swin-T frozen (full frame)":>38s} | {"0.3%":>10s} | {"10.6%":>10s}')
print(f'{"v1 EfficientNetV2 (full frame)":>38s} | {"0.3%":>10s} | {"10.1%":>10s}')
print(f'{"v1 Ensemble (full frame)":>38s} | {"0.3%":>10s} | {"9.3%":>10s}')
print('-'*65)

for name, res in [('v2 Swin-T frozen (face-crop)', swin_results),
                   ('v2 EffNetV2 frozen (face-crop)', effnet_results)]:
    eng = res['engagement']['accuracy_4class'] * 100
    avg = np.mean([m['accuracy_4class'] for m in res.values()]) * 100
    print(f'{name:>38s} | {eng:>9.1f}% | {avg:>9.1f}%')

if e2e_results:
    eng = e2e_results['engagement']['accuracy_4class'] * 100
    avg = np.mean([m['accuracy_4class'] for m in e2e_results.values()]) * 100
    print(f'{"v2 E2E Swin-T (face-crop)":>38s} | {eng:>9.1f}% | {avg:>9.1f}%')

eng = ensemble_results['engagement']['accuracy_4class'] * 100
avg = np.mean([m['accuracy_4class'] for m in ensemble_results.values()]) * 100
print(f'{"v2 Mega-Ensemble":>38s} | {eng:>9.1f}% | {avg:>9.1f}%')

print('-'*65)
print(f'{"CORAL (NB1, OpenFace AUs)":>38s} | {"48.3%":>10s} | {"—":>10s}')
print(f'{"Selim (EfficientNetB7)":>38s} | {"67.5%":>10s} | {"—":>10s}')
print(f'{"ViBED-Net (SOTA)":>38s} | {"73.4%":>10s} | {"—":>10s}')

# NB6 v2 Summary

## What Changed from v1 → v2

| Fix | v1 (0.3% acc) | v2 (target 40-55%) |
|-----|---------------|---------------------|
| **Input** | Full frame (desk, wall, room) | **MTCNN face crop** (face only) |
| **Loss** | Weighted CE (over-corrected → inverted collapse) | **Focal Loss γ=2** (down-weights easy examples) |
| **Sampling** | Random (99% majority class in batch) | **WeightedRandomSampler** (balanced batches) |
| **Augmentation** | Weak (temporal jitter only) | **Strong** (H-flip, color jitter, affine, grayscale, MixUp) |
| **E2E Fine-tuning** | Disabled | **Enabled** (AMP, differential LR, OneCycleLR) |
| **Backbone learns** | Nothing (frozen) | **Face expression features** via E2E |

## Key Insight
> Engagement is expressed through **facial micro-expressions** (AUs), **gaze shifts**, and **head pose**.
> ImageNet backbones on full frames see furniture, not faces.
> Face cropping is the **single most important fix** — it ensures the backbone processes the right signal.

## Files Saved (for NB7 fusion)
- `proba_*4c_{dim}.npy` — 4-class probabilities (N, 4) for SOTA comparison
- `proba_*_{dim}.npy` — Binary P(High) = P(class≥2) for v5 fusion compatibility
- `e2e_swin_{dim}.pt` — Fine-tuned model weights
- `face_bboxes.pkl` — Cached MTCNN face bounding boxes

## Next Steps
1. **Run NB7** fusion with these improved probabilities
2. If E2E accuracy > 50%, consider fine-tuning EfficientNet too
3. Report v1→v2 improvement as ablation study in paper